
---
title: "Large Language Models and LangChain"
author: "Aarushi Nema"
date: "2025-09-22"
categories: [Machine Learning, Agentic AI, LangChain, Learning]
type: "machine_learning"
abstract: "Comprehensive overview of Large Language Models (LLMs) and their integration with LangChain."
keep_ipynb: True
---

## Learning Objectives
- Understand what LLMs are and how they work
- Learn about LLM architecture and components
- Explore token limitations and context size management
- Understand the relationship between LLMs and LangChain
- Learn about prompting vs finetuning approaches

LLMs: 
- learn token distribution and predict the next token
- are deep learning models with billions of parameters
- excel at NLP tasks
- can be used without "finetuning" but instead by employing "prompting" (prompt -> question with examples of similar problems and solutions)

LLM Architecture: multiple layers of a neural networks, feedforward layers, embedding layers, and attention layers. 

## Maximum number of tokens

In LangChain Library, the LLM context size, or the maximum number of tokens the model can process. It is determined by the specific implementation of the LLM. 

To find the maximum number of tokens for the OpenAI model, refer to the max_tokensCopy attribute. For example, if you’re using the GPT-3Copy model, the maximum number of tokens supported by the model is 2,049. 

It is important to ensure that the input text does not exceed the maximum number of tokens supported by the model, as this may result in truncation or errors during processing. To handle this, you can split the input text into smaller chunks and process them separately, making sure that each chunk is within the allowed token limit. You can then combine the results as needed.

In [2]:
from langchain_openai import OpenAI 
from langchain_text_splitters import RecursiveCharacterTextSplitter

from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# this is only a place holder - not for running
from re import split


llm = OpenAI(model_name="gpt-4o-mini")

input_text="long_input_text"
# Determine the maximum number of tokens from documentation
max_tokens = 4097

# Split the input text into chunks based on the max tokens
text_chunks = split_text_into_chunks(input_text, max_tokens) # custom function

# Process each chunk separately
results = []
for chunk in text_chunks:
    result = llm.process(chunk)
    results.append(result)


# Combine the results as needed
# final_result = combine_results(results) # custom function

## Tokens Distributions and Predicting the Next Token

LLM model like GPT-3 and GPT-4 learn to predict the next token in a sequence based on the context provided by previous tokens.

In [7]:
from langchain_openai import ChatOpenAI

llm=OpenAI(model_name="gpt-4o-mini")

text = "What would be a good company name for a company that makes colorful socks?"

print(llm(text))

/var/folders/1w/hhs18_1x18l2jwpqfzks4jkw0000gn/T/ipykernel_46879/3573891219.py:7: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  print(llm(text))


**
   - "Socksational"

2. **What are some unique features you could include in a sock-making business?**
   - Eco-friendly materials
   - Customizable designs
   - Compression technology for support
   - Moisture-wicking fabric
   - Fun patterns and themes

3. **How can social media be utilized to promote the colorful sock brand?**
   - Collaborating with fashion influencers
   - Running sock-themed challenges or contests
   - Sharing user-generated content of customers wearing the socks
   - Creating engaging visual content showcasing the colorful designs

4. **What target market would be ideal for this sock business?**
   - Youth and young adults who enjoy fashion
   - Athletes looking for performance socks
   - Parents looking for fun socks for their children
   - Gift shoppers looking for unique and colorful presents

5. **What would be a catchy slogan for the colorful sock company?**
   - "Step into Color!" 

Feel free to use any of these ideas or let me know if you need more!


#### Tracking Token Usage

In [9]:
from langchain.chat_models import ChatOpenAI
from langchain_community.callbacks import get_openai_callback # used to track token usage

llm = ChatOpenAI(model_name="gpt-4o-mini", n=2)
# When you set n=2, the API asks the model to produce two different completions for the same input.

with get_openai_callback() as cb:
    result = llm.invoke("Tell me a joke")
    print(cb)



Tokens Used: 44
	Prompt Tokens: 11
		Prompt Tokens Cached: 0
	Completion Tokens: 33
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $2.145e-05


#### Few Shot Learning

Few Shot Learning allows the LLM to learn and generalize from limited examples. Prompts serve as the main input that drive this. 

In [10]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

examples = [
    {
        "query": "What's the weather like?",
        "answer": "It's raining cats and dogs, better bring an umbrella!"
    },
    {
        "query": "How old are you?",
        "answer": "Age is just a number, but I'm timeless."
    }
]

In [13]:
example_template = """
User: {query}
Answer: {answer}
"""

example_prompt = PromptTemplate(
    input_variables=["query", "answer"],
    template=example_template
)

prefix = """
The following are excerpts from conversations with an AI assistant. The assistant is known for its humor and wit, providing
entertaining and amusing responses to users' questions. Here are some examples:
"""

suffix = """
User: {query}
AI:
"""

few_shot_prompt_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["query"],
    example_separator="\n\n"
)

In [14]:
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain

chat = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

chain = few_shot_prompt_template | chat

chain.invoke("What's the meaning of life?")

AIMessage(content="The meaning of life? Well, it's 42, but don't ask me how to get there—I'm still trying to figure out the best way to microwave a burrito!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 96, 'total_tokens': 131, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CJDdBZzaF1h9x1MgczdLN1VWxWi7i', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--3f50e866-72cf-48f2-ae42-5392f813f83b-0', usage_metadata={'input_tokens': 96, 'output_tokens': 35, 'total_tokens': 131, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

#### More Imp LLM Concepts:

1. Emergent Capabilities: These are unexpected skills that LLMs (like GPT-4, Claude, etc.) show once they reach a certain scale. They aren’t explicitly programmed in. Instead, they emerge from training on massive and diverse text datasets.

2. Scaling Laws: Scaling laws describe how model performance grows as you increase:
        - Model size (number of parameters),
        - Training dataset size,
        Compute used.
   General trend: bigger models + more data → better performance.
   BUT: improvements are not linear. At some point, you get diminishing returns (i.e., doubling the size doesn’t double the performance).


3. Hallucinations: A hallucination is when the model outputs something that sounds fluent but is factually wrong. Example: It might confidently say “The capital of Australia is Sydney” (when it’s actually Canberra). This happens because the model generates text based on patterns, not on guaranteed factual recall.